# W11-D3 概念实验：Connector 覆盖率

核心概念来自 Markdown：Channel、MCP Client、Webhook、Outbound System Bridge 分散在不同子系统，没有统一 Connector 抽象。用能力维度计算覆盖率，区分“传输可用”和“企业系统治理完整”。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False


In [ ]:
systems = ["Channel", "MCP Client", "Webhook", "Outbound Bridge"]
dimensions = ["连接建立", "认证/授权", "重试/健康", "审计", "私网系统"]
coverage = np.array([
    [1, 1, 1, 1, 0],  # 入站消息成熟，但不是企业系统连接
    [1, 1, 0, 0, 0],  # transport 支持，治理能力不足
    [1, 1, 1, 1, 0],  # 事件出站
    [0, 0, 0, 0, 1],  # 目标设计覆盖私网，但 Phase-0 未完成
], dtype=int)
print("系统覆盖率:")
for name, row in zip(systems, coverage):
    print(f"{name:16s} {row.mean():.0%}")
print("总体维度覆盖:", dict(zip(dimensions, coverage.mean(axis=0))))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.imshow(coverage, cmap="Blues", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(dimensions)), dimensions, rotation=25, ha="right")
ax.set_yticks(range(len(systems)), systems)
ax.set_title("Connector 子系统能力覆盖矩阵")
for i in range(coverage.shape[0]):
    for j in range(coverage.shape[1]):
        ax.text(j, i, "✓" if coverage[i, j] else "·", ha="center", va="center", fontsize=13)
plt.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
row_rates = coverage.mean(axis=1)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(systems, row_rates, color=["#4c78a8", "#f2cf5b", "#59a14f", "#e15759"])
ax.set_ylim(0, 1.05)
ax.set_ylabel("覆盖率")
ax.set_title("从“能连接”到“可治理连接”的覆盖率")
ax.tick_params(axis="x", rotation=20)
for i, value in enumerate(row_rates):
    ax.text(i, value + 0.03, f"{value:.0%}", ha="center")
plt.tight_layout()
plt.show()
plt.close(fig)
print("结论：Outbound Bridge 的目标能力尚未形成可运行覆盖；统一 Connector 应复用认证、健康检查、审计和私网边界，而不是再造一个传输客户端。")